# MACURA learns a Backflip

Pogo (a one-legged pogo-stick gymnast) learns to backflip. A backflip is an
aggressive, unstable maneuver: a **fixed-horizon** rollout (MBPO) over-imagines the
flight and Pogo face-plants, while **MACURA** truncates the imagined rollout when
the model gets uncertain and learns the flip from trustworthy pieces.

We train four algorithms (MACURA, MBPO, M2AC, SAC) that share the SAC backbone and
the ensemble — only the rollout strategy differs — then save the policies and watch
them fly. Run top to bottom on a **GPU** runtime.

### Setup: Drive, device, token, clone

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, warnings
os.environ['MUJOCO_GL'] = 'egl'
warnings.filterwarnings('ignore')
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

In [ ]:
TOKEN = None
try:
    from google.colab import userdata
    TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    pass
if not TOKEN:
    import getpass
    TOKEN = getpass.getpass('GitHub token (hidden): ')
os.environ['GITHUB_TOKEN'] = TOKEN

In [ ]:
%%bash
set -e
curl -sH "Authorization: token ${GITHUB_TOKEN}" \
  "https://raw.githubusercontent.com/silvergjeka22/macura-backflip/main/bash/setup_colab.sh" \
  -o /content/setup_colab.sh
bash /content/setup_colab.sh

### Imports + config (single source of truth = `configs/macura_backflip.yaml`)

In [ ]:
import sys
if '/content/macura-backflip' not in sys.path:
    sys.path.insert(0, '/content/macura-backflip')
import copy, json, yaml
import numpy as np
from IPython.display import Video, display
from envs import pogo_env
from training import train as trainer
from viz import plots

def deep_merge(base, over):
    out = copy.deepcopy(base)
    for k, v in over.items():
        out[k] = deep_merge(out[k], v) if isinstance(v, dict) and isinstance(out.get(k), dict) else copy.deepcopy(v)
    return out

CFG = yaml.safe_load(open('/content/macura-backflip/configs/macura_backflip.yaml'))
cfg = deep_merge(CFG, CFG['profiles']['fast'])      # the 20k-step run (edit seeds in the yaml to go faster)
DRIVE = cfg['experiment']['drive_root']
PLOTS = f'{DRIVE}/plots'; os.makedirs(PLOTS, exist_ok=True)
print('steps', cfg['experiment']['total_env_steps'], '| seeds', cfg['experiment']['seeds'], '| ->', DRIVE)

### Meet Pogo (untrained — it just flails)

In [ ]:
env, obs_dim, act_dim = pogo_env.make_env(cfg['env'], seed=0, render=True)
plots.render_filmstrip(env, policy='random', title='Random Pogo', save_path=f'{PLOTS}/pogo.png');

### Train the four algorithms
Idempotent — a run whose log already exists on Drive is skipped and reloaded. Each
eval line prints return, flips, landing rate, and faceplant rate.

In [ ]:
runs = []
for algo in cfg['experiment']['algorithms']:
    for seed in cfg['experiment']['seeds']:
        lp = f'{DRIVE}/logs/{algo}_seed{seed}.json'
        if os.path.exists(lp):
            print('skip (exists):', algo, seed); runs.append(json.load(open(lp)))
        else:
            runs.append(trainer.train_one(algo, cfg, DRIVE, seed=seed))

### Results
Sample efficiency, backflips landed (headline), faceplant rate, final quality, and
MACURA's signature (rollout length grows / kappa falls).

In [ ]:
plots.plot_sample_efficiency(runs, save_path=f'{PLOTS}/sample_eff.png')
plots.plot_success_rate(runs, save_path=f'{PLOTS}/success.png')
plots.plot_failure_rate(runs, save_path=f'{PLOTS}/faceplant.png')
plots.plot_final_quality(runs, save_path=f'{PLOTS}/final.png')
plots.plot_rollout_depth([r for r in runs if r['algo'] == 'macura'][0], save_path=f'{PLOTS}/macura_depth.png');

### Flight demo — watch each best policy

In [ ]:
videos = trainer.record_best_videos(cfg, DRIVE, DEVICE, seeds=cfg['experiment']['seeds'], seconds=8)
for name, path in videos.items():
    print(name); display(Video(path, embed=True, width=360))

In [ ]:
labels = {name.split('_seed')[0]: path for name, path in videos.items()}
if labels:
    display(Video(plots.stitch_videos_grid(labels, f'{DRIVE}/videos/reel.mp4', cols=2), embed=True, width=520))

### Measured ordering

In [ ]:
for r in sorted(runs, key=lambda r: -r['final_eval']['eval_return']):
    fe = r['final_eval']
    print(f"{r['algo']:7s} seed{r['seed']}  return {fe['eval_return']:7.1f}"
          f"  land {fe.get('eval_success_rate', 0):.2f}  fail {fe.get('eval_failure_rate', 0):.2f}")
print('\nWatch them live on your Mac:  mjpython run_live_mac.py --compare runs/checkpoints')